# Catalog query and stream lifecycle

**Audience:** readers who know how to create and scan an Archiver catalog and want to understand what query and stream objects own.

**Prerequisites:** the public `Catalog.files()` and `Catalog.duplicates()` query APIs.

**Learning goals:** distinguish catalogs, queries, streams, and handles; observe lazy execution; safely interleave and close streams; reuse queries; and understand scan-pinned duplicate handles. All examples use public APIs and disposable synthetic files.

## Outline

1. Build and print a small catalog.
2. Relate the catalog, immutable queries, and cursor-backed streams.
3. Reuse and interleave file, group, and member streams.
4. Retain a duplicate handle across outer-stream closure and refresh.
5. Order every file by duplicate-group size.
6. Close resources and review the lifecycle rules.

## 1. Disposable catalog

The fixture contains a four-copy nested group, a three-copy group, two empty duplicates, and one singleton.

In [ ]:
import sqlite3
from collections import Counter
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog
from archiver.query import content, size

workspace = TemporaryDirectory(prefix="archiver-stream-lifecycle-")
base = Path(workspace.name)
root = base / "source"
root.mkdir()

for folder, payload, count in (
    ("large/nested", b"LARGE", 4),
    ("medium", b"MID", 3),
    ("empty", b"", 2),
):
    directory = root / folder
    directory.mkdir(parents=True)
    for number in range(count):
        (directory / f"copy-{number}.bin").write_bytes(payload)
(root / "singleton.txt").write_text("only one", encoding="utf-8")

catalog = Catalog.create(base / "catalog.sqlite")
catalog.scan_directory(root)

First print the generated filesystem and then the catalog's complete current observation set. `files(root)` includes singletons and duplicates alike.

In [ ]:
print("Generated filesystem")
for path in sorted(root.rglob("*")):
    relative = path.relative_to(root).as_posix()
    suffix = "/" if path.is_dir() else f"  {path.stat().st_size} bytes"
    print(f"  {relative}{suffix}")

print("\nCatalog current observations")
with closing(catalog.files(root).stream_all()) as observations:
    for observation in observations:
        print(
            f"  {observation.relative_path.as_posix():<30} "
            f"{observation.size_bytes:>2} bytes  "
            f"{observation.content_id.digest[:10]}..."
        )

## 2. Catalog, query, stream, and cursor

A `Catalog` owns one open SQLite connection. A query is an immutable description and owns no cursor. Calling a stream terminal returns a lazy generator; advancing that generator for the first time executes its SQL and opens that stream's cursor. Each active stream has an independent cursor, while all streams from the catalog share its connection.

```text
Catalog
  └─ SQLite connection
       ├─ current-file stream cursor
       ├─ duplicate-group stream cursor
       └─ member stream cursor

Query
  └─ immutable description; no cursor
```

Cursor ownership is an implementation resource contract; SQL and cursors are not exposed through the public API.

Constructing and transforming a query performs no observation read. The query can be reused, and a transformation returns a new value rather than mutating the original.

In [ ]:
file_query = catalog.files(root).sort(size.desc()).limit(3)
larger_preview = file_query.limit(5)

assert file_query is not larger_preview
assert len(file_query.fetch()) == 3
assert len(larger_preview.fetch()) == 5

In [ ]:
# Example listing all files in a stream
for item in larger_preview.fetch():
    print(item.relative_path)

## 3. One query, multiple streams

Each terminal call evaluates the reusable query independently. Exhausting a stream reaches its cleanup automatically; `closing()` also protects partial-consumption paths.

In [ ]:
with closing(file_query.stream()) as first_stream:
    first_paths = [row.relative_path.as_posix() for row in first_stream]
with closing(file_query.stream()) as second_stream:
    second_paths = [row.relative_path.as_posix() for row in second_stream]

assert first_paths == second_paths
print("Reusable query result:", first_paths)

## 4. Independent group and member cursors

Advancing a duplicate-group stream does not fetch its members. A member cursor opens only when its own stream advances. Because cursor positions are independent, the outer group stream can advance while a member stream is active.

In [ ]:
duplicate_query = catalog.duplicates(root)

with closing(duplicate_query.stream_all()) as groups:
    first_group = next(groups)
    with closing(first_group.stream_all_members()) as members:
        first_member = next(members)
        second_group = next(groups)

print("First group: ", first_group.content_id.digest[:10] + "...")
print("First member:", first_member.relative_path)
print("Second group:", second_group.content_id.digest[:10] + "...")
assert first_member.content_id == first_group.content_id
assert second_group.content_id != first_group.content_id

Closing the outer stream does not invalidate an already selected handle. The handle can open a fresh member stream as long as the catalog remains open.

In [ ]:
outer_stream = duplicate_query.stream_all()
retained_group = next(outer_stream)
outer_stream.close()

with closing(retained_group.stream_all_members()) as members:
    retained_paths_before_refresh = [member.relative_path.as_posix() for member in members]

assert len(retained_paths_before_refresh) == retained_group.file_instance_count == 4
print("Retained members:", retained_paths_before_refresh)

## 5. A handle pins observed membership

A group stream resolves one successful scan on first advancement. Its handles keep using that scan even after a later refresh. A separate terminal may resolve the newer scan. This pins catalog observations—not the live filesystem and not authority to modify it.

In [ ]:
changed_path = root / "large" / "nested" / "copy-3.bin"
changed_path.write_bytes(b"CHANGED")
catalog.scan_directory(root)

with closing(retained_group.stream_all_members()) as old_members:
    pinned_paths = [member.relative_path.as_posix() for member in old_members]
current_count = catalog.files(root).where(content.is_(retained_group.content_id)).count()

assert len(pinned_paths) == 4
assert current_count == 3
print(f"Pinned handle: {len(pinned_paths)} members; latest scan: {current_count} matching files")

## 6. Exercise: show duplicates and singletons together

Order every current file by the number of paths sharing its content: largest duplicate groups first and singletons last. Predict the copy-count sequence before running the answer. This intentionally materializes the small tutorial catalog in Python; use streaming rather than this pattern for an archive too large to retain in memory.

In [ ]:
with closing(catalog.files(root).stream_all()) as files:
    all_files = tuple(files)

copy_count = Counter(file.content_id for file in all_files)
ordered_files = sorted(
    all_files,
    key=lambda file: (
        -copy_count[file.content_id],
        file.content_id.algorithm,
        file.content_id.digest,
        file.relative_path.as_posix(),
    ),
)

for file in ordered_files:
    count = copy_count[file.content_id]
    label = "singleton" if count == 1 else f"{count} copies"
    print(f"  [{label:>9}] {file.relative_path.as_posix():<30} {file.content_id.digest[:10]}...")

ordered_counts = [copy_count[file.content_id] for file in ordered_files]
assert ordered_counts == sorted(ordered_counts, reverse=True)
assert ordered_counts[-1] == 1

--------------------------------------------
Alternative solution with pandas (quite raw)

In [ ]:
import pandas as pd

with closing(catalog.files(root).stream_all()) as files:
    all_files = tuple(files)
    
all_ids = [F.content_id.digest for F in all_files]
all_filenames = [F.relative_path.as_posix() for F in all_files]
copy_count = Counter(file.content_id.digest for file in all_files)
tbl=pd.DataFrame({'id':all_ids, 'fname':all_filenames})
# Group by 'id', calculate the group size, and assign it back as a column
tbl['count'] = tbl.groupby('id')['id'].transform('count')
# Sort descending by 'count', then ascending (alphabetically) by 'id'
tbl = tbl.sort_values(by=['count', 'id'], ascending=[False, True])
# Truncate the 'id' column to the first 10 characters before printing
tbl['id'] = tbl['id'].str[:10]+'...'
print(tbl)

## 7. Closing the catalog

The catalog connection must remain open until every stream and retained handle has finished. A query remains an ordinary immutable value after catalog closure, but advancing a new stream from it fails because execution needs the closed connection.

In [ ]:
catalog.close()

with closing(file_query.stream()) as closed_catalog_stream:
    try:
        next(closed_catalog_stream)
    except sqlite3.ProgrammingError as error:
        print("Expected closed-catalog error:", error)
    else:
        raise AssertionError("advancing a stream should require an open catalog")

workspace.cleanup()

## Lifecycle summary and pitfalls

| Object | Opens a cursor? | Lifetime rule |
| --- | --- | --- |
| `Catalog` | No permanent cursor; owns the connection | Keep open through all stream work |
| Query | No | Immutable and reusable; each terminal evaluates separately |
| Stream generator | On first advancement | Exhaust it or close it explicitly |
| Duplicate handle | No eager member cursor | Can open fresh member streams pinned to its selected scan |

Common pitfalls: a plain `break` does not guarantee prompt generator closure; `stream_all()` is intentionally unbounded; closing an outer stream does not close a separately opened member stream; and collecting a stream into a tuple gives up its constant-Python-memory benefit.

## References

- [Plan 004.3 — Materialization and Reductions](../../docs/plans/004.3-materialization-and-reductions.md) introduced bounded and explicit unbounded current-file streams.
- [Plan 004.4.1 — Duplicate Group Sorting and Streaming](../../docs/plans/004.4.1-duplicate-group-sorting-and-streaming.md) introduced group/member streams, independent cursor behavior, retained handles, and scan pinning.
- [Public query grammar](../../docs/query-grammar.md) is the current API reference.